# weight-decay-decoupled — faded example 3: Skip decoupled decay when wd=0

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-decoupled`. The last cell reports your progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-decoupled`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-decoupled"
DD_SUBTOPIC = "Optimizer: decoupled weight decay (AdamW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When `wd=0`, the decoupled weight-decay step `p.mul_(1 - lr * 0)` would multiply by 1.0 — a no-op. While mathematically equivalent, explicitly guarding with `if wd != 0` avoids an unnecessary in-place multiply and makes the code's intent clearer: weight decay only runs when it's actually configured.

## Faded exercise 3

Implement `adamw_step_with_guard(p, grad, m, v, lr, beta1, beta2, eps, wd, step)`. Add an `if wd != 0` guard around the weight decay step — skip it when `wd` is zero. The rest of the AdamW logic (moment update, bias correction, Adam update) always runs.

Complete the blank that conditionally applies the weight-decay step.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def adamw_step_with_guard(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    if wd != 0:
        p.mul_(1 - lr * wd)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

t.manual_seed(9)
p0 = t.tensor([1.0])
g = t.tensor([0.3])
m = t.zeros(1); v = t.zeros(1)
adamw_step_with_guard(p0, g, m, v, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.0, step=1)
print('wd=0 step, p:', p0.item())


import torch as t

def adamw_step_with_guard(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    if wd != 0:
        p.mul_(1 - lr * wd)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

def _test():
    # wd=0: result should match standard Adam (no decay)
    t.manual_seed(0)
    p_test = t.tensor([0.5])
    p_ref = t.tensor([0.5])
    g = t.tensor([0.1])
    m_t = t.zeros(1); v_t = t.zeros(1)
    m_r = t.zeros(1); v_r = t.zeros(1)
    lr, b1, b2, eps = 1e-3, 0.9, 0.999, 1e-8
    adamw_step_with_guard(p_test, g, m_t, v_t, lr, b1, b2, eps, wd=0.0, step=1)
    # plain Adam reference
    p_ref_opt = t.tensor([0.5], requires_grad=True)
    opt = t.optim.Adam([p_ref_opt], lr=lr, betas=(b1, b2), eps=eps)
    opt.zero_grad(); p_ref_opt.grad = t.tensor([0.1]); opt.step()
    assert t.allclose(p_test, p_ref_opt.detach(), atol=1e-6)
    # wd>0: result should match AdamW
    t.manual_seed(0)
    p_test2 = t.tensor([0.5])
    m2 = t.zeros(1); v2 = t.zeros(1)
    adamw_step_with_guard(p_test2, g, m2, v2, lr, b1, b2, eps, wd=0.01, step=1)
    p_ref2 = t.tensor([0.5], requires_grad=True)
    opt2 = t.optim.AdamW([p_ref2], lr=lr, betas=(b1, b2), eps=eps, weight_decay=0.01)
    opt2.zero_grad(); p_ref2.grad = t.tensor([0.1]); opt2.step()
    assert t.allclose(p_test2, p_ref2.detach(), atol=1e-6)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def adamw_step_with_guard(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    if wd != 0:
        p.mul_(1 - lr * wd)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

t.manual_seed(9)
p0 = t.tensor([1.0])
g = t.tensor([0.3])
m = t.zeros(1); v = t.zeros(1)
adamw_step_with_guard(p0, g, m, v, lr=1e-2, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.0, step=1)
print('wd=0 step, p:', p0.item())
```
</details>